In [1]:
import numpy as np
import pickle
import gzip
import awkward as ak
import re
import yaml

import correctionlib

from coffea import lookup_tools
from coffea.lookup_tools import txt_converters, rochester_lookup

from topcoffea.modules.paths import topcoffea_path
from ttbarEFT.modules.paths import ttbarEFT_path

In [42]:
def GetBtagEff(year, jets, wp='medium'):
    # similar to GetMCeffFunc in topeft.modules.corrections
    if year not in clib_year_map.keys():
        raise Exception(f"Error: Unknown year \"{year}\".")

    pathToBtagMCeff = ttbarEFT_path('data/btagSF/UL/btagMCeff_%s.pkl.gz'%year)
    hists = {}
    with gzip.open(pathToBtagMCeff) as fin:
        hists = pickle.load(fin)['btag']

    h = hists['jetptetaflav']
    hnum = h[{'WP': wp}]
    hden = h[{'WP': 'all'}]

    eff = hnum/hden
    eff_lookup = lookup_tools.dense_lookup.dense_lookup(
        eff.values(), 
        [
            eff.axes['jpt'].edges,
            eff.axes['jeta'].edges,
            eff.axes['flavour'].edges
        ]
    )

    # fun = lambda pt, abseta, flav: eff_lookup(pt,abseta,flav)
    # return fun
    return eff_lookup(jets.pt, np.abs(jets.eta), jets.hadronFlavour)

In [49]:
# pathToBtagMCeff = ttbarEFT_path('data/btagSF/UL/btagMCeff_2017.pkl.gz')
pathToBtagMCeff = "btagEff_040326.pkl.gz"
hists = {}
with gzip.open(pathToBtagMCeff) as fin:
    hists = pickle.load(fin)['btag']

h = hists['jetptetaflav']
hnum = h[{'WP': 'medium'}]
hden = h[{'WP': 'all'}]

eff = hnum/hden
eff_lookup = lookup_tools.dense_lookup.dense_lookup(
    eff.values(), 
    [ax.edges for ax in eff.axes]
)

In [50]:
hists['jetptetaflav'].axes

(StrCategory(['all', 'medium'], growth=True, name='WP'),
 IntCategory([0, 4, 5], name='flavour', label='jet flavour (int)'),
 Variable([20, 30, 50, 70, 100, 140, 200, 300, 600, 1000], name='jpt', label='Jet p_{T} [GeV]'),
 Variable([0, 0.6, 1.2, 2.4], name='jeta', label='Jet \\eta [GeV]'))

In [48]:
eff_lookup

<coffea.lookup_tools.dense_lookup.dense_lookup object at 0x7f54a66e6140> 3 dimensional histogram with axes:
	1: [0. 1. 2. 3.]
	2: [  20.   30.   50.   70.  100.  140.  200.  300.  600. 1000.]
	3: [0.  0.6 1.2 2.4]

In [32]:
eff[{'flavour':5}].values()

array([[       nan,        nan,        nan],
       [0.82297104, 0.8092884 , 0.76016955],
       [0.84541783, 0.83494422, 0.78678449],
       [0.85777555, 0.84766617, 0.79964242],
       [0.8655145 , 0.85414612, 0.80690578],
       [0.86925504, 0.85973723, 0.81352232],
       [0.86970193, 0.86198369, 0.80777622],
       [0.85655478, 0.84106174, 0.77658277],
       [0.7719715 , 0.76491228, 0.60769231]])

In [33]:
eff[{'flavour':4}].values()

array([[       nan,        nan,        nan],
       [0.18057626, 0.1757595 , 0.16313298],
       [0.16432799, 0.16108097, 0.15092929],
       [0.17151431, 0.16324011, 0.15051581],
       [0.1773282 , 0.17198068, 0.15034729],
       [0.18179822, 0.16921423, 0.15889755],
       [0.19921814, 0.19195262, 0.18165567],
       [0.23571599, 0.22830189, 0.20206986],
       [0.25142857, 0.2775    , 0.22509225]])

In [34]:
eff[{'flavour':0}].values()

array([[       nan,        nan,        nan],
       [0.01021316, 0.01138698, 0.01588699],
       [0.00721268, 0.0082103 , 0.01181431],
       [0.00681906, 0.00828717, 0.01110622],
       [0.00570327, 0.00743053, 0.01141253],
       [0.006207  , 0.00791045, 0.01374602],
       [0.007591  , 0.01005324, 0.01980744],
       [0.01670084, 0.01828867, 0.03954728],
       [0.04539749, 0.05506855, 0.08821254]])

In [41]:
eff_lookup(5, 900, )

np.float64(0.6076923076923076)